# 03 阅读材料分类：从眼动特征到行为识别

**数据来源：** 同 Notebook 01/02，PyMovements ToyDataset（EyeLink 1000 Hz，4 段文本 × 5 页 = 20 trials）。

**研究问题：** 仅凭眼动特征，能否自动区分被试正在阅读哪段文本？

**本 Notebook 涵盖：**
1. 特征提取 → 构建分类数据集（`text_id` 作为 4 类标签）
2. 多模型对比：RandomForest / GradientBoosting / SVM / LogisticRegression
3. Leave-One-Out 交叉验证（样本量仅 20，不适合随机划分）
4. 混淆矩阵可视化 + 分类报告
5. 排列重要性分析：哪些眼动特征对分类贡献最大
6. 人因解读与方法局限性讨论

> 应用场景：在终端人因评测中，眼动特征分类可用于自动识别用户的交互模式（如浏览 vs 搜索 vs 阅读），为界面设计提供数据驱动的优化依据（Duchowski, 2017）。

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

from pathlib import Path
import pymovements as pm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.inspection import permutation_importance

from gaze_toolkit.pymovements_adapter import from_pymovements
from gaze_toolkit.preprocess import preprocess
from gaze_toolkit.events import attach_events
from gaze_toolkit.features import extract_features

# ── 加载数据集并批量提取特征 ─────────────────────────────────────────────
DATA_PATH = Path('..').resolve() / '.cache' / 'pm_gt_probe' / 'ToyDataset'
ds = pm.Dataset('ToyDataset', path=DATA_PATH)
ds.load()

fileinfo = ds.fileinfo['gaze'].to_pandas()
fileinfo = fileinfo.drop_duplicates(subset=['text_id', 'page_id']).reset_index(drop=True)

rows = []
for idx in fileinfo.index:
    text_id = int(fileinfo.loc[idx, 'text_id'])
    page_id = int(fileinfo.loc[idx, 'page_id'])
    rec = from_pymovements(ds.gaze[idx], sampling_rate_hz=1000.0)
    rec_clean = preprocess(rec)
    rec_ev = attach_events(rec_clean)
    feats = extract_features(rec_ev)
    feats['text_id'] = text_id
    feats['page_id'] = page_id
    rows.append(feats)

df = pd.DataFrame(rows)
feature_cols = [c for c in df.columns if c not in ('text_id', 'page_id')]

# 去除零方差特征
feature_cols = [c for c in feature_cols if df[c].std() > 1e-8]

X = df[feature_cols].fillna(0).values
y = df['text_id'].values

print(f'分类数据集：{X.shape[0]} 样本 × {X.shape[1]} 特征，{len(np.unique(y))} 类')
print(f'类别分布：{dict(zip(*np.unique(y, return_counts=True)))}')
print(f'标签含义：text_id ∈ {{0,1,2,3}}，每个 text 有 5 页（5 trials）')

## 1. Leave-One-Out 交叉验证：多模型对比

样本量仅 20，传统 train/test split 会导致测试集过小、结果不稳定。采用 **Leave-One-Out (LOO) 交叉验证**：每次留 1 个 trial 做测试，其余 19 个训练。重复 20 次后汇总，这是小样本分类的标准做法。

对比 4 个经典分类器，展示 `gaze_toolkit.modeling` 模块支持的模型种类。

In [ ]:
# ── LOO 交叉验证，4 个分类器 ─────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

MODELS = {
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
}

loo = LeaveOneOut()
results = {}

for name, model in MODELS.items():
    y_pred = cross_val_predict(model, X_scaled, y, cv=loo)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average='macro')
    results[name] = {
        'accuracy': acc,
        'f1_macro': f1,
        'y_pred': y_pred,
    }
    print(f'{name:25s}  Accuracy={acc:.1%}  F1(macro)={f1:.3f}')

# 选择最佳模型
best_name = max(results, key=lambda k: results[k]['f1_macro'])
print(f'\n最佳模型: {best_name} (F1={results[best_name]["f1_macro"]:.3f})')

# ── 柱状图对比 ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
model_names = list(results.keys())
accs = [results[n]['accuracy'] for n in model_names]
f1s = [results[n]['f1_macro'] for n in model_names]
x_pos = np.arange(len(model_names))

bars1 = ax.bar(x_pos - 0.18, accs, 0.35, label='Accuracy', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x_pos + 0.18, f1s, 0.35, label='F1 (macro)', color='#FF5722', alpha=0.8)

# 标注数值
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.0%}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Score')
ax.set_title('LOO 交叉验证：4 模型分类性能对比（4 类文本，20 trials）', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.15)
ax.axhline(0.25, color='gray', ls='--', lw=1, alpha=0.5, label='随机基线 (25%)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/nb03_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. 混淆矩阵与分类报告

用最佳模型的 LOO 预测结果，展示各文本的分类准确性和混淆模式。

In [ ]:
# ── 混淆矩阵 ─────────────────────────────────────────────────────────────
y_pred_best = results[best_name]['y_pred']
cm = confusion_matrix(y, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 子图 1：混淆矩阵热力图
ax = axes[0]
disp = ConfusionMatrixDisplay(cm, display_labels=[f'Text {i}' for i in range(4)])
disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
ax.set_title(f'{best_name} 混淆矩阵（LOO, n=20）', fontsize=11)

# 子图 2：各类别的 precision / recall / f1
ax2 = axes[1]
report = classification_report(y, y_pred_best, target_names=[f'Text {i}' for i in range(4)],
                                output_dict=True)
class_metrics = pd.DataFrame({
    'Precision': [report[f'Text {i}']['precision'] for i in range(4)],
    'Recall': [report[f'Text {i}']['recall'] for i in range(4)],
    'F1': [report[f'Text {i}']['f1-score'] for i in range(4)],
}, index=[f'Text {i}' for i in range(4)])

class_metrics.plot(kind='bar', ax=ax2, color=['#2196F3', '#FF5722', '#4CAF50'], alpha=0.8)
ax2.set_ylabel('Score')
ax2.set_title('各文本分类指标', fontsize=11)
ax2.set_ylim(0, 1.15)
ax2.legend(fontsize=9)
ax2.set_xticklabels(class_metrics.index, rotation=0)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../examples/nb03_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

# 文本报告
print(classification_report(y, y_pred_best,
                             target_names=[f'Text {i}' for i in range(4)]))

## 3. 排列重要性分析：哪些特征驱动分类？

排列重要性（Permutation Importance）通过打乱单个特征、观察准确率下降幅度来量化其贡献。相比内置 feature_importances_，排列重要性不受特征量纲和树结构偏差影响，更适合解释（Altmann et al., 2010）。

In [ ]:
# ── 在全量数据上训练最佳模型，然后计算排列重要性 ─────────────────────────
best_model = MODELS[best_name].__class__(**MODELS[best_name].get_params())
best_model.fit(X_scaled, y)

perm_imp = permutation_importance(best_model, X_scaled, y,
                                   n_repeats=20, random_state=42,
                                   scoring='accuracy')

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std,
}).sort_values('importance_mean', ascending=False)

# ── Top-15 排列重要性条形图 ───────────────────────────────────────────────
top_n = 15
top = imp_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = ['#FF5722' if v > 0.01 else '#9E9E9E' for v in top['importance_mean']]
ax.barh(range(top_n), top['importance_mean'], xerr=top['importance_std'],
        color=colors_imp, alpha=0.8, capsize=3)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top['feature'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('排列重要性（Accuracy 下降量）')
ax.set_title(f'{best_name} 排列重要性 Top-{top_n}（红色 = 高贡献）', fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../examples/nb03_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

# 人因解读
print('排列重要性 Top-5：')
for i, row in top.head(5).iterrows():
    print(f'  {row["feature"]:30s}  {row["importance_mean"]:.4f} +/- {row["importance_std"]:.4f}')

## 4. 误分类分析：哪些 trial 被误判？

查看被误分类的 trial，分析误判原因（是否与该 trial 的阅读时长、页码等特征有关）。

In [ ]:
# ── 误分类明细 ─────────────────────────────────────────────────────────────
misclassified = df.loc[y != y_pred_best, ['text_id', 'page_id',
                                            'duration_ms', 'fixation_count',
                                            'fixation_duration_mean',
                                            'saccade_amplitude_mean']].copy()
misclassified['predicted'] = y_pred_best[y != y_pred_best]
misclassified.columns = ['真实text', '页码', '时长(ms)', '注视数', '注视均时(ms)', '扫视幅度(px)', '预测text']

if len(misclassified) > 0:
    print(f'误分类 trial 数: {len(misclassified)} / {len(df)} ({len(misclassified)/len(df):.0%})\n')
    print(misclassified.to_string(index=False))

    # ── 可视化：误分类 trial 在 PCA 空间中的位置 ──────────────────────────
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

    for tid in range(4):
        mask = y == tid
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors[tid], s=60, alpha=0.6,
                   label=f'Text {tid}', edgecolors='white', linewidth=0.5)

    # 标记误分类点
    mis_mask = y != y_pred_best
    ax.scatter(X_pca[mis_mask, 0], X_pca[mis_mask, 1],
               facecolors='none', edgecolors='red', s=200, linewidths=2,
               label='误分类', zorder=5)
    for j in np.where(mis_mask)[0]:
        ax.annotate(f'T{y[j]}p{df.iloc[j]["page_id"]}',
                   (X_pca[j, 0], X_pca[j, 1]),
                   fontsize=8, color='red', fontweight='bold',
                   xytext=(5, 5), textcoords='offset points')

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title('误分类 trial 在 PCA 特征空间中的位置（红圈）', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../examples/nb03_misclassified_pca.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('所有 trial 均正确分类！LOO Accuracy = 100%')
    print('注意：这可能表明模型过拟合，或 4 段文本的眼动模式确实差异显著。')

## 5. 特征消融实验：最少需要多少特征？

从 Top-5 最重要特征开始，逐步增加特征数量，观察 LOO 准确率的变化曲线。这展示了"特征工程→ 效率分析"的完整思路。

In [ ]:
# ── 特征消融曲线 ──────────────────────────────────────────────────────────
sorted_features = imp_df['feature'].tolist()
n_tests = [1, 2, 3, 5, 8, 10, 15, 20, len(feature_cols)]
n_tests = sorted(set([n for n in n_tests if n <= len(feature_cols)]))

ablation_results = []
for n_feat in n_tests:
    selected = sorted_features[:n_feat]
    col_idx = [feature_cols.index(f) for f in selected]
    X_sub = X_scaled[:, col_idx]

    model = MODELS[best_name].__class__(**MODELS[best_name].get_params())
    y_pred_sub = cross_val_predict(model, X_sub, y, cv=loo)
    acc = accuracy_score(y, y_pred_sub)
    ablation_results.append({'n_features': n_feat, 'accuracy': acc})
    print(f'  Top-{n_feat:2d} 特征 → LOO Accuracy = {acc:.1%}')

abl_df = pd.DataFrame(ablation_results)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(abl_df['n_features'], abl_df['accuracy'], 'o-', lw=2,
        color='#2196F3', markersize=8, markerfacecolor='white', markeredgewidth=2)
ax.axhline(0.25, color='gray', ls='--', lw=1, alpha=0.5, label='随机基线')
ax.fill_between(abl_df['n_features'], 0.25, abl_df['accuracy'],
                alpha=0.1, color='#2196F3')
ax.set_xlabel('使用的特征数量（按排列重要性排序）')
ax.set_ylabel('LOO Accuracy')
ax.set_title(f'特征消融实验：{best_name} 分类准确率 vs 特征数量', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../examples/nb03_feature_ablation.png', dpi=120, bbox_inches='tight')
plt.show()

# 找到达到最高准确率 95% 的最少特征数
max_acc = abl_df['accuracy'].max()
threshold = max_acc * 0.95
sufficient = abl_df[abl_df['accuracy'] >= threshold].iloc[0]
print(f'\n达到最高准确率 95%（{threshold:.1%}）只需 Top-{int(sufficient["n_features"])} 个特征')

## 6. 小结与方法讨论

**本 Notebook 完成的工作：**

| 步骤 | 方法 | 意义 |
|---|---|---|
| 数据构建 | 20 trials, 42 维特征, 4 类标签 | 真实 EyeLink 数据，非模拟 |
| 交叉验证 | Leave-One-Out (LOO) | 小样本下的标准评估策略 |
| 多模型对比 | RF / GBDT / SVM / LR | 展示 modeling 模块的灵活性 |
| 混淆矩阵 | 分类报告 + 可视化 | 逐类精度分析 |
| 特征解释 | 排列重要性 Top-15 | 可解释 AI，适合人因评测汇报 |
| 特征消融 | 逐步增加特征 | 确定最小有效特征集 |

**关于分类性能的讨论：**
- 4 类文本分类的随机基线为 25%，模型显著超过基线说明眼动特征包含文本区分信息
- LOO 交叉验证比 holdout 更充分利用了有限样本，但可能高估泛化性能
- 单被试 20 trials 的结果不可推广到群体水平——这是原型展示，不是发表级证据

**与研究/应用场景的关联：**
- **自动化评测**：眼动分类器可自动识别参与者在不同 UI 界面或研究任务场景下的行为模式
- **特征工程**：排列重要性分析帮助研究人员选择最具判别力的指标，缩短人因工程研究和用户体验研究的分析周期
- **可解释性**：传统 ML 模型（RF/GBDT）比深度学习更适合向研究团队或业务团队汇报人因洞察
- **端到端 pipeline**：从数据加载到模型评估全链路自动化，适合研究原型验证与应用分析流程

**方法局限性：**
- n=20 样本量过小，统计功效不足
- 同一被试的 5 个页面间不独立（violation of i.i.d.），但 LOO 无法处理分组结构
- 未做超参数搜索（样本太少，搜索容易过拟合）
- 缺少外部验证集

**改进方向：**
- 采集多名被试数据后，改用 Leave-One-Subject-Out 交叉验证
- 加入瞳孔特征（认知负荷维度，需更换设备）
- 添加 AOI 特征（兴趣区域注视比例），需要刺激材料的布局标注